In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Datasets" / 'AusGrid2013'
import pandas as pd
import numpy as np
from tqdm import tqdm 
import matplotlib.pyplot as plt
import hierarchicalforecast
from hierarchicalforecast.utils import aggregate
df1 = pd.read_csv(str(DATA_DIR / "2012-2013 Solar home electricity data v2.csv"), low_memory=False)
df1.columns = df1.iloc[0,:].values
df1 = df1.iloc[1:, :-1].reset_index(drop = True) # all values of column "row quality" are non
df = df1[df1['Consumption Category'] == 'GC']
b = [len(df[df['Customer'] == f"{i}"]) == 365 for i in tqdm(range(1, 301))] #check length
df = pd.concat([df[df['Customer'] == f"{int(i)+1}"] for i in tqdm(np.where(np.array(b))[0])])

In [ ]:
df

In [ ]:
id_vars = ['Customer', 'Generator Capacity', 'Postcode', 'Consumption Category', 'date']
time_cols = [col for col in df.columns if col not in id_vars]
df_long = df.melt(
    id_vars=id_vars,
    value_vars=time_cols,
    var_name='time',
    value_name='consumption'
)
# df_long = df_long[df_long['Consumption Category'] == 'GC']
a = df_long['date']+' '+df_long['time']
a = pd.to_datetime(a, format='%d/%m/%Y %H:%M')
df_long['ds'] = a
dff = df_long.drop(columns = ['date','time', 'Generator Capacity']).copy()
del df_long
dff['Customer'] = dff['Customer'].astype('int') #.sort_values()
dff = dff.sort_values(by=['Customer','ds'], ascending=True)
dff = dff.reset_index(drop = True)

In [ ]:
dff.columns = ["Customer", "Postcode","Consumption Category", "y", "ds"]
dff['y'] = dff['y'].astype('float')
SPEC = [
    ["Consumption Category"],  # Level 1
    ["Consumption Category", "Postcode"],  # Level 2
    ["Consumption Category", "Postcode", "Customer"]
]
Y_df, S_df, tags = aggregate(dff, SPEC)
#Y_df['levels'] = Y_df['unique_id'].str.count('/') + 1

In [ ]:
arr = Y_df['unique_id'].unique()

label2uid = {label: i for i, label in enumerate(arr)}
uid2label = {i: label for label, i in label2uid.items()} 

codes, uniques = pd.factorize(arr)   # codes: 每个元素对应的整数编码
label2id = dict(zip(uniques, range(len(uniques))))

Y_df['partition_id'] = Y_df['unique_id'].map(label2id)

#Y_df = Y_df[Y_df['ds'] >= '2017-04-01']

Y_df

In [ ]:
Y_df.to_csv(str(DATA_DIR / "AusGrid2013_univariate.csv"), index = False)
S_df.to_csv(str(DATA_DIR / 'AusGrid_S.csv'), index = False)
import pickle
with open(str(DATA_DIR / "tags.pkl"), "wb") as f:
    pickle.dump(tags, f)

# Check

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
import matplotlib.pyplot as plt
import hierarchicalforecast
from hierarchicalforecast.utils import aggregate
df = pd.read_csv(str(DATA_DIR / "2012-2013 Solar home electricity data v2.csv"), low_memory=False)
df.columns = df.iloc[0,:].values
df = df.iloc[1:, :-1].reset_index(drop = True) # all values of column "row quality" are non
id_vars = ['Customer', 'Generator Capacity', 'Postcode', 'Consumption Category', 'date']
time_cols = [col for col in df.columns if col not in id_vars]
df_long = df.melt(
    id_vars=id_vars,
    value_vars=time_cols,
    var_name='time',
    value_name='consumption'
)
df_long = df_long[df_long['Consumption Category'] == 'GC']
a = df_long['date']+' '+df_long['time']
a = pd.to_datetime(a, format='%d/%m/%Y %H:%M')
df_long['ds'] = a
dff = df_long.drop(columns = ['date','time', 'Generator Capacity']).copy()
#del df_long
dff['Customer'] = dff['Customer'].astype('int') #.sort_values()
dff = dff.sort_values(by=['Customer','ds'], ascending=True)
dff = dff.reset_index(drop = True)

In [ ]:
a = df[df['Consumption Category'] == 'GC']

In [ ]:

b

In [ ]:
np.array(b)